# Construire la couche Gold : contrat initial des arrêts

## Objectif concret

Ce notebook initialise la construction du dataset Gold Indusense. Il couvre uniquement les trois premières sections de la roadmap : le vocabulaire, la règle temporelle et la définition V1 d'un arrêt machine. Il construit ensuite la table logique `gold_stop_event` en mémoire.

**Décision déjà prise :** la version `emergency-stop-v1` définit un arrêt comme un incident Silver dont `is_emergency_stop = true`. Cette version est volontairement restrictive : les arrêts manuels possibles, la criticité, les maintenances réactives et la production nulle seront étudiés dans une version ultérieure.

**Ce notebook ne crée encore aucun label et ne persiste aucune donnée Gold.** La table `gold_stop_event` est un DataFrame pandas reproductible à partir de Silver ; l'écriture physique viendra plus tard dans la roadmap.

In [1]:
from pathlib import Path

from tabulate import tabulate

PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / 'pyproject.toml').exists(), (
    'Ouvrir le notebook depuis la racine du projet indusense.'
)

print(f'Projet : {PROJECT_ROOT.name}')
print('Périmètre : sections 1 à 3 de la roadmap Gold ; table logique en mémoire, sans écriture en base.')


Projet : indusense
Périmètre : sections 1 à 3 de la roadmap Gold ; table logique en mémoire, sans écriture en base.


## 1. Employer le vocabulaire du dataset temporel

Une **ancre de prédiction** est l'instant `t` où le modèle devrait décider. Une **feature** est une information connue à cet instant ou avant. Un **label** est la réponse future que le modèle apprend pendant l'entraînement.

Une ligne Gold sera un **snapshot de features** : la photographie d'une machine à l'instant `t`. Une **fenêtre passée** fournit les features ; un **horizon** de 6 h, 12 h ou 24 h fournit le label.

In [2]:
glossary = [
    ['ancre de prédiction', 'instant t où une décision est demandée'],
    ['feature', 'information disponible à t ou avant'],
    ['label', 'réponse future apprise pendant l’entraînement'],
    ['fenêtre passée', 'période avant t utilisée pour les features'],
    ['horizon', 'période future à prédire : 6 h, 12 h ou 24 h'],
    ['fuite de données', 'utilisation d’une information inconnue à t'],
]

print(tabulate(glossary, headers=['Terme', 'Sens dans Gold'], tablefmt='github'))


| Terme               | Sens dans Gold                                |
|---------------------|-----------------------------------------------|
| ancre de prédiction | instant t où une décision est demandée        |
| feature             | information disponible à t ou avant           |
| label               | réponse future apprise pendant l’entraînement |
| fenêtre passée      | période avant t utilisée pour les features    |
| horizon             | période future à prédire : 6 h, 12 h ou 24 h  |
| fuite de données    | utilisation d’une information inconnue à t    |


## 2. Fixer la règle temporelle avant tout calcul

Les features peuvent utiliser les données dans `[t - fenêtre ; t]`. Pour un horizon `h`, le label utilise uniquement les événements situés dans `(t ; t + h]` : l'ancre est exclue et la borne finale est incluse.

Cette convention évite la **fuite de données** : un événement futur ne peut jamais devenir une feature. Elle implique aussi qu'un snapshot ne pourra servir à l'entraînement que si tout son horizon futur est disponible.

In [3]:
import pandas as pd

prediction_at = pd.Timestamp('2026-02-10 10:00:00', tz='UTC')
horizon = pd.Timedelta(hours=6)
candidate_stop_times = pd.DatetimeIndex([
    prediction_at,
    prediction_at + pd.Timedelta(hours=1),
    prediction_at + horizon,
])

# La convention (t ; t + h] exclut l'événement exactement à l'ancre.
inside_horizon = (candidate_stop_times > prediction_at) & (candidate_stop_times <= prediction_at + horizon)
temporal_example = pd.DataFrame({
    'stop_at': candidate_stop_times,
    'retenu_pour_target_stop_6h': inside_horizon,
})

assert inside_horizon.tolist() == [False, True, True]
print(f'Ancre : {prediction_at}')
print(tabulate(temporal_example, headers='keys', tablefmt='github', showindex=False))


Ancre : 2026-02-10 10:00:00+00:00
| stop_at                   | retenu_pour_target_stop_6h   |
|---------------------------|------------------------------|
| 2026-02-10 10:00:00+00:00 | False                        |
| 2026-02-10 11:00:00+00:00 | True                         |
| 2026-02-10 16:00:00+00:00 | True                         |


## 3. Formaliser le contrat V1 de l'arrêt machine

Un **contrat de label** indique sans ambiguïté quels événements deviennent positifs. Pour Gold V1, un `machine_stop` provient exclusivement de `silver.incident` lorsque `is_emergency_stop = true`.

La table logique `gold_stop_event` conservera la machine, la date, l'identifiant de l'incident source, la table source et la version de définition. Les commentaires servent à qualifier les cas ; ils ne créent pas le label et ne sont pas recopiés dans Gold. Les maintenances réactives, les incidents de criticité élevée et les productions nulles sont exclus de V1, en attente d'une règle métier pour les arrêts manuels.

In [4]:
import os

from sqlalchemy import text

from indusense.db.engine import create_database_engine

# Les paramètres sont chargés sans afficher de valeur sensible.
for raw_line in (PROJECT_ROOT / '.docker' / '.env').read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip())

contract = [
    ['source_table', 'silver.incident'],
    ['selection_rule', 'is_emergency_stop = true'],
    ['machine_id', 'machine_code'],
    ['stop_at', 'occurred_at'],
    ['source_event_id', 'incident_id'],
    ['stop_definition_version', 'emergency-stop-v1'],
]

engine = create_database_engine()
with engine.connect() as connection:
    candidate_count = connection.scalar(
        text('SELECT COUNT(*) FROM silver.incident WHERE is_emergency_stop')
    )

assert candidate_count > 0, 'Le contrat V1 ne sélectionne aucun événement Silver.'
print(tabulate(contract, headers=['Élément du contrat', 'Valeur V1'], tablefmt='github'))
print(f'Incidents Silver actuellement sélectionnés par V1 : {candidate_count}')
print('Validation : contrat lisible et données sources présentes ; la cellule suivante construit gold_stop_event.')

engine.dispose()


| Élément du contrat      | Valeur V1                |
|-------------------------|--------------------------|
| source_table            | silver.incident          |
| selection_rule          | is_emergency_stop = true |
| machine_id              | machine_code             |
| stop_at                 | occurred_at              |
| source_event_id         | incident_id              |
| stop_definition_version | emergency-stop-v1        |
Incidents Silver actuellement sélectionnés par V1 : 19
Validation : contrat lisible et données sources présentes ; la cellule suivante construit gold_stop_event.


### 3.1 Construire la table logique `gold_stop_event`

Le **grain** de cette table est : une ligne par incident reconnu comme arrêt par `emergency-stop-v1`. La requête sélectionne uniquement les arrêts d'urgence, renomme les colonnes Silver selon le contrat Gold et ajoute les constantes de lignée.

La **lignée** (*lineage*) permet de retrouver l'origine d'une donnée. Ici, `source_table` et `source_event_id` permettent de revenir à l'incident Silver complet. Nous conservons aussi l'indicateur ayant déclenché la règle et la criticité pour auditer le résultat, sans recopier le commentaire libre.

La cellule vérifie les colonnes obligatoires, l'absence de valeurs manquantes, l'unicité des événements sources et le respect intégral de la définition V1. Elle ne crée aucune table PostgreSQL et aucun fichier.

In [5]:
stop_event_query = text(
    """
    SELECT
        machine_code AS machine_id,
        occurred_at AS stop_at,
        'emergency-stop-v1' AS stop_definition_version,
        incident_id AS source_event_id,
        'silver.incident' AS source_table,
        is_emergency_stop AS source_is_emergency_stop,
        severity AS source_severity
    FROM silver.incident
    WHERE is_emergency_stop
    ORDER BY machine_code, occurred_at, incident_id
    """
)

# La table Gold est construite en mémoire à partir de la source Silver.
engine = create_database_engine()
with engine.connect() as connection:
    gold_stop_event = pd.read_sql(stop_event_query, connection)
engine.dispose()

required_columns = [
    'machine_id',
    'stop_at',
    'stop_definition_version',
    'source_event_id',
    'source_table',
]

# Ces assertions transforment le contrat métier en contrôles exécutables.
assert len(gold_stop_event) == candidate_count
assert gold_stop_event[required_columns].notna().all().all()
assert gold_stop_event['source_event_id'].is_unique
assert gold_stop_event['source_is_emergency_stop'].all()
assert gold_stop_event['stop_definition_version'].eq('emergency-stop-v1').all()
assert gold_stop_event['source_table'].eq('silver.incident').all()

print('Aperçu des événements d’arrêt V1 :')
print(tabulate(gold_stop_event.head(10), headers='keys', tablefmt='github', showindex=False))
print(f'Nombre d’événements : {len(gold_stop_event)}')
print(f'Événements sources uniques : {gold_stop_event["source_event_id"].nunique()}')
print('Validation : gold_stop_event respecte le grain, la lignée et la définition emergency-stop-v1.')


Aperçu des événements d’arrêt V1 :
| machine_id   | stop_at                   | stop_definition_version   | source_event_id   | source_table    | source_is_emergency_stop   |   source_severity |
|--------------|---------------------------|---------------------------|-------------------|-----------------|----------------------------|-------------------|
| MACH-01      | 2025-06-17 00:07:00+00:00 | emergency-stop-v1         | INC-000055        | silver.incident | True                       |                 4 |
| MACH-02      | 2025-12-18 12:10:00+00:00 | emergency-stop-v1         | INC-000667        | silver.incident | True                       |                 4 |
| MACH-03      | 2025-12-26 16:25:00+00:00 | emergency-stop-v1         | INC-000690        | silver.incident | True                       |                 4 |
| MACH-04      | 2025-11-15 04:56:00+00:00 | emergency-stop-v1         | INC-000548        | silver.incident | True                       |                 4 |
| MAC

### 3.2 Formaliser les trois labels et leurs bornes temporelles

Pour une ligne `(machine_id, prediction_at)`, chaque label répond à une question binaire : **la machine de cette ligne connaîtra-t-elle un arrêt V1 dans les 6 h, 12 h ou 24 h suivantes ?** Les événements d'autres machines ne modifient pas ce label. Leurs informations connues à `prediction_at` pourront être étudiées plus tard comme features de contexte complexes.

La convention commune est `(prediction_at ; prediction_at + horizon]` : l'instant de prédiction est exclu et la borne de fin est incluse. Un arrêt dans les 6 h rend donc aussi positifs les horizons 12 h et 24 h.

Gold V1 conservera uniquement les instants pour lesquels les 24 heures futures sont entièrement observables. La grille réelle de `prediction_at` sera créée à la section 4 ; cette cellule valide d'abord le contrat sur des cas jouets contrôlés.

In [6]:
LABEL_HORIZONS_HOURS = (6, 12, 24)

def compute_stop_labels(stop_events, machine_id, prediction_at):
    same_machine_stops = stop_events.loc[
        stop_events['machine_id'].eq(machine_id), 'stop_at'
    ]
    return {
        f'target_stop_{hours}h': int(
            (
                (same_machine_stops > prediction_at)
                & (same_machine_stops <= prediction_at + pd.Timedelta(hours=hours))
            ).any()
        )
        for hours in LABEL_HORIZONS_HOURS
    }

prediction_at = pd.Timestamp('2026-01-01T10:00:00Z')
test_cases = [
    ('borne t exclue', 'MACH-Y', prediction_at, (0, 0, 0)),
    ('borne 6 h incluse', 'MACH-Y', prediction_at + pd.Timedelta(hours=6), (1, 1, 1)),
    ('après 6 h', 'MACH-Y', prediction_at + pd.Timedelta(hours=6, seconds=1), (0, 1, 1)),
    ('borne 12 h incluse', 'MACH-Y', prediction_at + pd.Timedelta(hours=12), (0, 1, 1)),
    ('borne 24 h incluse', 'MACH-Y', prediction_at + pd.Timedelta(hours=24), (0, 0, 1)),
    ('après 24 h', 'MACH-Y', prediction_at + pd.Timedelta(hours=24, seconds=1), (0, 0, 0)),
    ('autre machine', 'MACH-X', prediction_at + pd.Timedelta(hours=1), (0, 0, 0)),
]

test_results = []
for case_name, event_machine, stop_at, expected in test_cases:
    toy_events = pd.DataFrame({'machine_id': [event_machine], 'stop_at': [stop_at]})
    labels = compute_stop_labels(toy_events, 'MACH-Y', prediction_at)
    actual = tuple(labels[f'target_stop_{hours}h'] for hours in LABEL_HORIZONS_HOURS)
    assert actual == expected, f'{case_name}: attendu {expected}, obtenu {actual}'
    assert actual[0] <= actual[1] <= actual[2]
    test_results.append([case_name, stop_at, event_machine, *actual])

# Un snapshot est admissible si son horizon maximal est entièrement observable.
assert prediction_at + pd.Timedelta(hours=24) <= prediction_at + pd.Timedelta(hours=24)
assert not (prediction_at + pd.Timedelta(hours=24) <= prediction_at + pd.Timedelta(hours=23, minutes=59))

print(tabulate(
    test_results,
    headers=['Cas testé', 'stop_at', 'machine événement', '6 h', '12 h', '24 h'],
    tablefmt='github',
))
print('Validation : bornes temporelles, même machine, horizons imbriqués et observabilité 24 h respectés.')

| Cas testé          | stop_at                   | machine événement   |   6 h |   12 h |   24 h |
|--------------------|---------------------------|---------------------|-------|--------|--------|
| borne t exclue     | 2026-01-01 10:00:00+00:00 | MACH-Y              |     0 |      0 |      0 |
| borne 6 h incluse  | 2026-01-01 16:00:00+00:00 | MACH-Y              |     1 |      1 |      1 |
| après 6 h          | 2026-01-01 16:00:01+00:00 | MACH-Y              |     0 |      1 |      1 |
| borne 12 h incluse | 2026-01-01 22:00:00+00:00 | MACH-Y              |     0 |      1 |      1 |
| borne 24 h incluse | 2026-01-02 10:00:00+00:00 | MACH-Y              |     0 |      0 |      1 |
| après 24 h         | 2026-01-02 10:00:01+00:00 | MACH-Y              |     0 |      0 |      0 |
| autre machine      | 2026-01-01 11:00:00+00:00 | MACH-X              |     0 |      0 |      0 |
Validation : bornes temporelles, même machine, horizons imbriqués et observabilité 24 h respectés.


## 4. Préparer la grille des instants de prédiction

### 4.1 Valider la cadence horaire

La **cadence de prédiction** est l'intervalle entre deux instants auxquels le modèle devra produire une estimation. Gold V1 retient une prédiction par machine et par heure, alignée sur les timestamps de `silver.telemetry`. Ce choix évite de dupliquer artificiellement une même information et conserve la précision nécessaire aux horizons de 6 h et 12 h.

Avant de construire la grille, la cellule suivante vérifie que chaque machine possède réellement une télémétrie horaire continue. Elle calcule aussi la dernière prédiction admissible, située 24 h avant la dernière mesure, et le nombre de lignes attendu. Elle ne construit pas encore `gold_prediction_grid`.

In [7]:
PREDICTION_CADENCE = pd.Timedelta(hours=1)
MAX_LABEL_HORIZON = pd.Timedelta(hours=24)

cadence_query = text(
    """
    WITH ordered AS (
        SELECT
            machine_code,
            measured_at,
            LAG(measured_at) OVER (
                PARTITION BY machine_code ORDER BY measured_at
            ) AS previous_at
        FROM silver.telemetry
    )
    SELECT
        machine_code,
        COUNT(*) AS measure_count,
        MIN(measured_at) AS first_measure,
        MAX(measured_at) AS last_measure,
        COUNT(*) FILTER (
            WHERE previous_at IS NOT NULL
              AND measured_at - previous_at <> INTERVAL '1 hour'
        ) AS non_hourly_intervals
    FROM ordered
    GROUP BY machine_code
    ORDER BY machine_code
    """
)

engine = create_database_engine()
with engine.connect() as connection:
    cadence_by_machine = pd.read_sql(cadence_query, connection)
engine.dispose()

assert not cadence_by_machine.empty
assert cadence_by_machine['first_measure'].nunique() == 1
assert cadence_by_machine['last_measure'].nunique() == 1
assert cadence_by_machine['measure_count'].nunique() == 1

first_prediction_at = cadence_by_machine['first_measure'].min()
last_observed_at = cadence_by_machine['last_measure'].max()
last_prediction_at = last_observed_at - MAX_LABEL_HORIZON
prediction_count_per_machine = int(
    (last_prediction_at - first_prediction_at) / PREDICTION_CADENCE
) + 1
expected_grid_rows = prediction_count_per_machine * len(cadence_by_machine)

cadence_summary = [
    ['machines', len(cadence_by_machine)],
    ['cadence', '1 heure'],
    ['première prédiction', first_prediction_at],
    ['dernière mesure observable', last_observed_at],
    ['dernière prédiction admissible', last_prediction_at],
    ['prédictions par machine attendues', prediction_count_per_machine],
    ['lignes de grille attendues', expected_grid_rows],
]

print(tabulate(cadence_summary, headers=['Contrôle', 'Valeur'], tablefmt='github'))
print('Validation : la cadence Gold horaire est compatible avec les séries Silver observées.')

| Contrôle                          | Valeur                    |
|-----------------------------------|---------------------------|
| machines                          | 15                        |
| cadence                           | 1 heure                   |
| première prédiction               | 2025-06-01 00:00:00+00:00 |
| dernière mesure observable        | 2026-06-08 23:00:00+00:00 |
| dernière prédiction admissible    | 2026-06-07 23:00:00+00:00 |
| prédictions par machine attendues | 8928                      |
| lignes de grille attendues        | 133920                    |
Validation : la cadence Gold horaire est compatible avec les séries Silver observées.


### 4.2 Construire `gold_prediction_grid` en mémoire

Le **grain** de la grille est une ligne par machine et par `prediction_at`. Pour chaque machine présente dans `silver.telemetry`, nous générons une série horaire régulière entre sa première mesure et sa dernière mesure moins 24 h. Cette limite garantit que les trois horizons futurs seront observables.

La grille est générée indépendamment des lignes de télémétrie : si une mesure venait à manquer, l'instant de prédiction resterait donc visible. À ce stade, Silver est continu ; la règle de traitement d'un éventuel manque sera discutée à l'opération suivante.

La cellule vérifie la continuité de la source, l'unicité de `(machine_id, prediction_at)`, la cadence d'une heure et l'observabilité des 24 h futures. Elle ne crée ni table PostgreSQL ni fichier.

In [8]:
# Ce calcul mesure les éventuels trous sans empêcher la création de la grille.
silver_expected_counts = (
    (cadence_by_machine['last_measure'] - cadence_by_machine['first_measure'])
    / PREDICTION_CADENCE
).astype(int) + 1
silver_missing_hour_count = int(
    (silver_expected_counts - cadence_by_machine['measure_count']).clip(lower=0).sum()
)

grid_parts = []
for machine in cadence_by_machine.itertuples(index=False):
    machine_prediction_times = pd.date_range(
        start=machine.first_measure,
        end=machine.last_measure - MAX_LABEL_HORIZON,
        freq=PREDICTION_CADENCE,
    )
    grid_parts.append(pd.DataFrame({
        'machine_id': machine.machine_code,
        'prediction_at': machine_prediction_times,
    }))

gold_prediction_grid = pd.concat(grid_parts, ignore_index=True)

assert len(gold_prediction_grid) == expected_grid_rows
assert gold_prediction_grid[['machine_id', 'prediction_at']].notna().all().all()
assert not gold_prediction_grid.duplicated(['machine_id', 'prediction_at']).any()
assert (
    gold_prediction_grid.groupby('machine_id')['prediction_at']
    .diff().dropna().eq(PREDICTION_CADENCE).all()
)
last_measure_by_machine = cadence_by_machine.set_index('machine_code')['last_measure']
observable_until = gold_prediction_grid['machine_id'].map(last_measure_by_machine)
assert (
    gold_prediction_grid['prediction_at'] + MAX_LABEL_HORIZON <= observable_until
).all()

grid_counts = (
    gold_prediction_grid.groupby('machine_id', as_index=False)
    .agg(
        predictions=('prediction_at', 'size'),
        first_prediction=('prediction_at', 'min'),
        last_prediction=('prediction_at', 'max'),
    )
)
print(tabulate(grid_counts, headers='keys', tablefmt='github', showindex=False))
print(f'Lignes de gold_prediction_grid : {len(gold_prediction_grid)}')
print('Validation : clé unique, cadence horaire et horizon futur de 24 h observable.')

| machine_id   |   predictions | first_prediction          | last_prediction           |
|--------------|---------------|---------------------------|---------------------------|
| MACH-01      |          8928 | 2025-06-01 00:00:00+00:00 | 2026-06-07 23:00:00+00:00 |
| MACH-02      |          8928 | 2025-06-01 00:00:00+00:00 | 2026-06-07 23:00:00+00:00 |
| MACH-03      |          8928 | 2025-06-01 00:00:00+00:00 | 2026-06-07 23:00:00+00:00 |
| MACH-04      |          8928 | 2025-06-01 00:00:00+00:00 | 2026-06-07 23:00:00+00:00 |
| MACH-05      |          8928 | 2025-06-01 00:00:00+00:00 | 2026-06-07 23:00:00+00:00 |
| MACH-06      |          8928 | 2025-06-01 00:00:00+00:00 | 2026-06-07 23:00:00+00:00 |
| MACH-07      |          8928 | 2025-06-01 00:00:00+00:00 | 2026-06-07 23:00:00+00:00 |
| MACH-08      |          8928 | 2025-06-01 00:00:00+00:00 | 2026-06-07 23:00:00+00:00 |
| MACH-09      |          8928 | 2025-06-01 00:00:00+00:00 | 2026-06-07 23:00:00+00:00 |
| MACH-10      |     

### 4.3 Aligner la grille avec la disponibilité de la télémétrie

Une jointure gauche (*left join*) conserve toutes les lignes de la grille, même lorsqu'aucune télémétrie Silver ne correspond à la même machine et à la même heure. La colonne booléenne `telemetry_row_available` indique cette correspondance.

Nous joignons uniquement `telemetry_id` pour vérifier l'existence de la ligne : les valeurs des capteurs et leurs `NULL` seront traités lors de la construction des features. Aucune imputation et aucune exclusion ne sont réalisées ici.

In [9]:
telemetry_key_query = text(
    """
    SELECT
        machine_code AS machine_id,
        measured_at AS prediction_at,
        telemetry_id
    FROM silver.telemetry
    WHERE measured_at BETWEEN :first_prediction_at AND :last_prediction_at
    """
)

engine = create_database_engine()
with engine.connect() as connection:
    telemetry_keys = pd.read_sql(
        telemetry_key_query,
        connection,
        params={
            'first_prediction_at': first_prediction_at,
            'last_prediction_at': last_prediction_at,
        },
    )
engine.dispose()

row_count_before_alignment = len(gold_prediction_grid)
gold_prediction_grid = gold_prediction_grid.merge(
    telemetry_keys,
    on=['machine_id', 'prediction_at'],
    how='left',
    validate='one_to_one',
)
gold_prediction_grid['telemetry_row_available'] = (
    gold_prediction_grid['telemetry_id'].notna()
)
gold_prediction_grid = gold_prediction_grid.drop(columns='telemetry_id')

assert len(gold_prediction_grid) == row_count_before_alignment
assert not gold_prediction_grid.duplicated(['machine_id', 'prediction_at']).any()
assert gold_prediction_grid['telemetry_row_available'].notna().all()

availability_counts = (
    gold_prediction_grid['telemetry_row_available']
    .value_counts(dropna=False)
    .rename_axis('telemetry_row_available')
    .reset_index(name='snapshots')
)
print(tabulate(availability_counts, headers='keys', tablefmt='github', showindex=False))
print(f'Heures Silver absentes détectées : {silver_missing_hour_count}')
print('Validation : tous les snapshots sont conservés et leur disponibilité est explicitée.')

| telemetry_row_available   |   snapshots |
|---------------------------|-------------|
| True                      |      133920 |
Heures Silver absentes détectées : 0
Validation : tous les snapshots sont conservés et leur disponibilité est explicitée.


### 4.4 Ajouter les trois labels à la grille

Pour chaque machine, une jointure temporelle recherche le premier `stop_at` strictement postérieur à `prediction_at`. Le label vaut `1` lorsque cet arrêt appartient à l'intervalle `(prediction_at ; prediction_at + horizon]`, sinon `0`.

Les trois labels sont stockés au format **large** dans `target_stop_6h`, `target_stop_12h` et `target_stop_24h`. Le futur timestamp d'arrêt sert uniquement au calcul du label : il n'est pas conservé dans `gold_prediction_grid` et ne pourra donc pas devenir accidentellement une feature.

In [10]:
label_columns = [f'target_stop_{hours}h' for hours in LABEL_HORIZONS_HOURS]
label_parts = []

for machine_id, machine_snapshots in gold_prediction_grid.groupby('machine_id', sort=False):
    snapshot_times = machine_snapshots[['prediction_at']].sort_values('prediction_at')
    machine_stops = (
        gold_stop_event.loc[gold_stop_event['machine_id'].eq(machine_id), ['stop_at']]
        .sort_values('stop_at')
        .rename(columns={'stop_at': 'next_stop_at'})
    )

    if machine_stops.empty:
        labelled_times = snapshot_times.copy()
        labelled_times['next_stop_at'] = pd.Series(
            pd.NaT,
            index=labelled_times.index,
            dtype=gold_stop_event['stop_at'].dtype,
        )
    else:
        labelled_times = pd.merge_asof(
            snapshot_times,
            machine_stops,
            left_on='prediction_at',
            right_on='next_stop_at',
            direction='forward',
            allow_exact_matches=False,
        )

    for hours in LABEL_HORIZONS_HOURS:
        labelled_times[f'target_stop_{hours}h'] = (
            labelled_times['next_stop_at'].notna()
            & (
                labelled_times['next_stop_at']
                <= labelled_times['prediction_at'] + pd.Timedelta(hours=hours)
            )
        ).astype('int8')

    labelled_times['machine_id'] = machine_id
    label_parts.append(labelled_times[['machine_id', 'prediction_at', *label_columns]])

label_values = pd.concat(label_parts, ignore_index=True)
row_count_before_labels = len(gold_prediction_grid)
gold_prediction_grid = gold_prediction_grid.merge(
    label_values,
    on=['machine_id', 'prediction_at'],
    how='left',
    validate='one_to_one',
)

assert len(gold_prediction_grid) == row_count_before_labels
assert gold_prediction_grid[label_columns].notna().all().all()
assert all(set(gold_prediction_grid[column].unique()) <= {0, 1} for column in label_columns)
assert (gold_prediction_grid['target_stop_6h'] <= gold_prediction_grid['target_stop_12h']).all()
assert (gold_prediction_grid['target_stop_12h'] <= gold_prediction_grid['target_stop_24h']).all()
assert 'next_stop_at' not in gold_prediction_grid.columns
assert not gold_prediction_grid.duplicated(['machine_id', 'prediction_at']).any()

label_summary = pd.DataFrame({
    'label': label_columns,
    'positifs': [int(gold_prediction_grid[column].sum()) for column in label_columns],
    'taux_positif': [gold_prediction_grid[column].mean() for column in label_columns],
})
print(tabulate(
    label_summary,
    headers='keys',
    tablefmt='github',
    showindex=False,
    floatfmt=('', 'd', '.4%'),
))
print('Validation : labels binaires, horizons imbriqués, futur non conservé et grain inchangé.')

| label           |   positifs |   taux_positif |
|-----------------|------------|----------------|
| target_stop_6h  |        102 |        0.0762% |
| target_stop_12h |        204 |        0.1523% |
| target_stop_24h |        408 |        0.3047% |
Validation : labels binaires, horizons imbriqués, futur non conservé et grain inchangé.


### 4.5 Ajouter les métadonnées d'exécution

Les métadonnées rendent chaque ligne traçable. Les versions et paramètres décrivent le contrat Gold ; `gold_build_run_id` et `gold_built_at` identifient l'exécution précise ; `source_observed_until` indique la borne Silver utilisée pour vérifier l'observabilité future.

Ces colonnes accompagnent le dataset mais ne sont pas des features. En particulier, `source_observed_until` peut être postérieur à `prediction_at` : l'utiliser pour entraîner le modèle provoquerait une fuite de données.

In [11]:
from uuid import UUID, uuid4

GOLD_DATASET_VERSION = 'gold-v1'
STOP_DEFINITION_VERSION = 'emergency-stop-v1'
gold_build_run_id = str(uuid4())
gold_built_at = pd.Timestamp.now(tz='UTC')

gold_prediction_grid['gold_dataset_version'] = GOLD_DATASET_VERSION
gold_prediction_grid['stop_definition_version'] = STOP_DEFINITION_VERSION
gold_prediction_grid['prediction_cadence_hours'] = 1
gold_prediction_grid['max_label_horizon_hours'] = max(LABEL_HORIZONS_HOURS)
gold_prediction_grid['gold_build_run_id'] = gold_build_run_id
gold_prediction_grid['gold_built_at'] = gold_built_at
gold_prediction_grid['source_observed_until'] = (
    gold_prediction_grid['machine_id'].map(last_measure_by_machine)
)

metadata_columns = [
    'gold_dataset_version',
    'stop_definition_version',
    'prediction_cadence_hours',
    'max_label_horizon_hours',
    'gold_build_run_id',
    'gold_built_at',
    'source_observed_until',
]
assert gold_prediction_grid[metadata_columns].notna().all().all()
assert gold_prediction_grid['gold_dataset_version'].eq(GOLD_DATASET_VERSION).all()
assert gold_prediction_grid['stop_definition_version'].eq(STOP_DEFINITION_VERSION).all()
assert gold_prediction_grid['prediction_cadence_hours'].eq(1).all()
assert gold_prediction_grid['max_label_horizon_hours'].eq(24).all()
assert gold_prediction_grid['gold_build_run_id'].nunique() == 1
assert gold_prediction_grid['gold_built_at'].nunique() == 1
assert str(UUID(gold_build_run_id)) == gold_build_run_id
assert (
    gold_prediction_grid['prediction_at'] + MAX_LABEL_HORIZON
    <= gold_prediction_grid['source_observed_until']
).all()

metadata_summary = [
    ['gold_dataset_version', GOLD_DATASET_VERSION],
    ['stop_definition_version', STOP_DEFINITION_VERSION],
    ['prediction_cadence_hours', 1],
    ['max_label_horizon_hours', max(LABEL_HORIZONS_HOURS)],
    ['gold_build_run_id', gold_build_run_id],
    ['gold_built_at', gold_built_at],
    ['source_observed_until', gold_prediction_grid['source_observed_until'].max()],
]
print(tabulate(metadata_summary, headers=['Métadonnée', 'Valeur'], tablefmt='github'))
print('Validation : métadonnées complètes, cohérentes et communes à cette exécution.')

| Métadonnée               | Valeur                               |
|--------------------------|--------------------------------------|
| gold_dataset_version     | gold-v1                              |
| stop_definition_version  | emergency-stop-v1                    |
| prediction_cadence_hours | 1                                    |
| max_label_horizon_hours  | 24                                   |
| gold_build_run_id        | b6a5b6fc-8aa6-46e6-9697-d494962a9fa9 |
| gold_built_at            | 2026-09-15 05:44:14.958288+00:00     |
| source_observed_until    | 2026-06-08 23:00:00+00:00            |
Validation : métadonnées complètes, cohérentes et communes à cette exécution.


## 5. Construire les features de télémétrie

### 5.1 Vérifier une fenêtre glissante avec une implémentation de référence

Une **fenêtre glissante** sélectionne, pour chaque `prediction_at`, les mesures récentes comprises ici dans `(prediction_at - 6 h ; prediction_at]`. La borne gauche est exclue et la mesure de `prediction_at` est incluse, car la prédiction est supposée être produite juste après sa réception. Avec une cadence horaire complète, six mesures sont donc attendues.

Avant la vectorisation, nous calculons explicitement les features de `temperature_c` pour trois snapshots de `MACH-01` précédant son premier arrêt V1. L'écart-type utilise `ddof=0` : il décrit toutes les valeurs disponibles dans la fenêtre, considérées comme la population observée. `feature_available_at` conserve la date source maximale réellement utilisée et doit rester inférieure ou égale à `prediction_at`.

In [12]:
REFERENCE_MACHINE_ID = 'MACH-01'
REFERENCE_WINDOW = pd.Timedelta(hours=6)
reference_stop_at = gold_stop_event.loc[
    gold_stop_event['machine_id'].eq(REFERENCE_MACHINE_ID), 'stop_at'
].min()
reference_last_prediction_at = reference_stop_at.floor('h')
reference_prediction_times = pd.DatetimeIndex([
    reference_last_prediction_at - pd.Timedelta(hours=6),
    reference_last_prediction_at - pd.Timedelta(hours=1),
    reference_last_prediction_at,
])

reference_telemetry_query = text(
    """
    SELECT measured_at, temperature_c
    FROM silver.telemetry
    WHERE machine_code = :machine_id
      AND measured_at > :load_start
      AND measured_at <= :load_end
    ORDER BY measured_at
    """
)
engine = create_database_engine()
with engine.connect() as connection:
    reference_telemetry = pd.read_sql(
        reference_telemetry_query,
        connection,
        params={
            'machine_id': REFERENCE_MACHINE_ID,
            'load_start': reference_prediction_times.min() - REFERENCE_WINDOW,
            'load_end': reference_prediction_times.max(),
        },
    )
engine.dispose()

expected_points = int(REFERENCE_WINDOW / PREDICTION_CADENCE)
reference_features = []
for prediction_at in reference_prediction_times:
    window_start = prediction_at - REFERENCE_WINDOW
    window_data = reference_telemetry.loc[
        reference_telemetry['measured_at'].gt(window_start)
        & reference_telemetry['measured_at'].le(prediction_at)
    ]
    available_temperature = window_data['temperature_c'].dropna()
    feature_available_at = window_data['measured_at'].max()
    missing_count = expected_points - len(available_temperature)

    assert len(window_data) <= expected_points
    assert feature_available_at <= prediction_at
    reference_features.append({
        'prediction_at': prediction_at,
        'window_start_excluded': window_start,
        'temperature_last': available_temperature.iloc[-1] if not available_temperature.empty else None,
        'temperature_mean_6h': available_temperature.mean(),
        'temperature_std_6h': available_temperature.std(ddof=0),
        'available_count_6h': len(available_temperature),
        'missing_count_6h': missing_count,
        'missing_rate_6h': missing_count / expected_points,
        'feature_available_at': feature_available_at,
    })

reference_feature_table = pd.DataFrame(reference_features)
assert len(reference_feature_table) == len(reference_prediction_times)
assert reference_feature_table['available_count_6h'].le(expected_points).all()
assert reference_feature_table['missing_rate_6h'].between(0, 1).all()
assert (
    reference_feature_table['feature_available_at']
    <= reference_feature_table['prediction_at']
).all()

print('Mesures chargées pour la référence :')
print(tabulate(reference_telemetry, headers='keys', tablefmt='github', showindex=False))
print('Features calculées explicitement :')
print(tabulate(reference_feature_table, headers='keys', tablefmt='github', showindex=False, floatfmt='.4f'))
print('Validation : chaque feature utilise uniquement la fenêtre passée (t - 6 h ; t].')

Mesures chargées pour la référence :
| measured_at               |   temperature_c |
|---------------------------|-----------------|
| 2025-06-16 13:00:00+00:00 |          61.674 |
| 2025-06-16 14:00:00+00:00 |          62.497 |
| 2025-06-16 15:00:00+00:00 |          63.031 |
| 2025-06-16 16:00:00+00:00 |          63.707 |
| 2025-06-16 17:00:00+00:00 |          67.146 |
| 2025-06-16 18:00:00+00:00 |          69.536 |
| 2025-06-16 19:00:00+00:00 |          72.378 |
| 2025-06-16 20:00:00+00:00 |          75.843 |
| 2025-06-16 21:00:00+00:00 |          78.239 |
| 2025-06-16 22:00:00+00:00 |          80     |
| 2025-06-16 23:00:00+00:00 |          80     |
| 2025-06-17 00:00:00+00:00 |          80     |
Features calculées explicitement :
| prediction_at             | window_start_excluded     |   temperature_last |   temperature_mean_6h |   temperature_std_6h |   available_count_6h |   missing_count_6h |   missing_rate_6h | feature_available_at      |
|---------------------------|---------

### 5.1.1 Comparer la référence à `pandas.rolling`

Une implémentation **vectorisée** applique une opération à une série entière plutôt que de parcourir chaque snapshot avec une boucle Python. `rolling('6h', closed='right')` représente ici la fenêtre `(t - 6 h ; t]`.

Nous recalculons les mêmes features, puis nous les comparons à la table de référence. `assert_frame_equal` accepte uniquement les minuscules écarts d'arrondi des nombres flottants ; une différence de sélection temporelle, de valeur manquante ou de formule ferait échouer la cellule.

In [13]:
def last_non_null(values):
    valid_values = values[~pd.isna(values)]
    return valid_values[-1] if len(valid_values) else float('nan')

reference_series = (
    reference_telemetry.set_index('measured_at')['temperature_c']
    .sort_index()
)
rolling_temperature = reference_series.rolling(
    window=REFERENCE_WINDOW,
    closed='right',
)
vectorized_features = pd.DataFrame(index=reference_series.index)
vectorized_features['temperature_last'] = rolling_temperature.apply(
    last_non_null, raw=True
)
vectorized_features['temperature_mean_6h'] = rolling_temperature.mean()
vectorized_features['temperature_std_6h'] = rolling_temperature.std(ddof=0)
vectorized_features['available_count_6h'] = rolling_temperature.count().astype(int)
vectorized_features['missing_count_6h'] = (
    expected_points - vectorized_features['available_count_6h']
)
vectorized_features['missing_rate_6h'] = (
    vectorized_features['missing_count_6h'] / expected_points
)
vectorized_features['feature_available_at'] = vectorized_features.index
vectorized_features = (
    vectorized_features.loc[reference_prediction_times]
    .rename_axis('prediction_at')
    .reset_index()
)

comparison_columns = [
    'temperature_last',
    'temperature_mean_6h',
    'temperature_std_6h',
    'available_count_6h',
    'missing_count_6h',
    'missing_rate_6h',
]
pd.testing.assert_series_equal(
    vectorized_features['prediction_at'],
    reference_feature_table['prediction_at'],
    check_names=False,
)
pd.testing.assert_series_equal(
    vectorized_features['feature_available_at'],
    reference_feature_table['feature_available_at'],
    check_names=False,
)
pd.testing.assert_frame_equal(
    vectorized_features[comparison_columns],
    reference_feature_table[comparison_columns],
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

maximum_differences = (
    vectorized_features[comparison_columns]
    .subtract(reference_feature_table[comparison_columns])
    .abs().max()
    .rename_axis('feature')
    .reset_index(name='écart_absolu_maximal')
)
print(tabulate(maximum_differences, headers='keys', tablefmt='github', showindex=False))
print('Validation : pandas.rolling reproduit la boucle de référence sur les trois snapshots.')

| feature             |   écart_absolu_maximal |
|---------------------|------------------------|
| temperature_last    |            0           |
| temperature_mean_6h |            0           |
| temperature_std_6h  |            4.66294e-14 |
| available_count_6h  |            0           |
| missing_count_6h    |            0           |
| missing_rate_6h     |            0           |
Validation : pandas.rolling reproduit la boucle de référence sur les trois snapshots.


### 5.1.2 Généraliser les features de télémétrie

Gold V1 calcule des features pour `temperature_c`, `pressure_bar`, `voltage_mean_v` et `rotation_mean_rpm`. Les fenêtres sont 3 h, 6 h, 12 h et 24 h. La fenêtre 1 h est volontairement écartée : avec une télémétrie horaire, ses statistiques répéteraient la dernière valeur.

Pour chaque capteur, `*_last` est la dernière valeur non nulle connue à `prediction_at`. Pour chaque fenêtre, nous calculons moyenne, minimum, maximum, écart-type de population, nombre de valeurs disponibles et taux de manque. L'étendue est déductible de `max - min`, les médianes et quantiles sont reportés, et aucun compteur d'outlier n'est créé car Silver ne fournit pas ce signal.

Toutes les opérations sont effectuées séparément par machine et avec `rolling(..., closed='right')` : aucune donnée postérieure à `prediction_at` ne peut entrer dans une feature. Les valeurs de début d'historique restent dans la grille et leur couverture réduite est visible par les comptes et taux de manque.

In [14]:
ROLLING_WINDOWS_HOURS = (3, 6, 12, 24)
TELEMETRY_FEATURE_SOURCES = {
    'temperature_c': 'temperature',
    'pressure_bar': 'pressure',
    'voltage_mean_v': 'voltage',
    'rotation_mean_rpm': 'rotation',
}

telemetry_feature_query = text(
    """
    SELECT
        machine_code AS machine_id,
        measured_at AS prediction_at,
        temperature_c,
        pressure_bar,
        voltage_mean_v,
        rotation_mean_rpm
    FROM silver.telemetry
    ORDER BY machine_code, measured_at
    """
)
engine = create_database_engine()
with engine.connect() as connection:
    telemetry_for_features = pd.read_sql(telemetry_feature_query, connection)
engine.dispose()

assert not telemetry_for_features.duplicated(['machine_id', 'prediction_at']).any()
telemetry_for_features = telemetry_for_features.sort_values(
    ['machine_id', 'prediction_at']
).reset_index(drop=True)
telemetry_by_time = telemetry_for_features.set_index('prediction_at')
feature_index = pd.MultiIndex.from_frame(
    telemetry_for_features[['machine_id', 'prediction_at']]
)
feature_index = feature_index.set_names(['machine_id', 'prediction_at'])
feature_values_by_column = {}

for source_column, feature_prefix in TELEMETRY_FEATURE_SOURCES.items():
    # La dernière valeur connue peut être antérieure à t si la mesure à t est NULL.
    feature_values_by_column[f'{feature_prefix}_last'] = (
        telemetry_for_features.groupby('machine_id')[source_column].ffill().to_numpy()
    )

    for hours in ROLLING_WINDOWS_HOURS:
        rolling_values = (
            telemetry_by_time.groupby('machine_id')[source_column]
            .rolling(window=f'{hours}h', closed='right')
        )
        rolling_statistics = pd.DataFrame({
            'mean': rolling_values.mean(),
            'min': rolling_values.min(),
            'max': rolling_values.max(),
            'std': rolling_values.std(ddof=0),
            'available_count': rolling_values.count(),
        })
        rolling_statistics.index = rolling_statistics.index.set_names(
            ['machine_id', 'prediction_at']
        )
        rolling_statistics = rolling_statistics.reindex(feature_index)
        available_count = rolling_statistics['available_count'].fillna(0).astype('int16')

        feature_values_by_column[f'{feature_prefix}_mean_{hours}h'] = (
            rolling_statistics['mean'].to_numpy()
        )
        feature_values_by_column[f'{feature_prefix}_min_{hours}h'] = (
            rolling_statistics['min'].to_numpy()
        )
        feature_values_by_column[f'{feature_prefix}_max_{hours}h'] = (
            rolling_statistics['max'].to_numpy()
        )
        feature_values_by_column[f'{feature_prefix}_std_{hours}h'] = (
            rolling_statistics['std'].to_numpy()
        )
        feature_values_by_column[f'{feature_prefix}_available_count_{hours}h'] = (
            available_count.to_numpy()
        )
        feature_values_by_column[f'{feature_prefix}_missing_rate_{hours}h'] = (
            1 - available_count.to_numpy() / hours
        )

telemetry_feature_values = (
    pd.DataFrame(feature_values_by_column, index=feature_index)
    .reset_index()
)
telemetry_feature_columns = [
    column for column in telemetry_feature_values.columns
    if column not in {'machine_id', 'prediction_at'}
]
assert len(telemetry_feature_columns) == 100
assert not telemetry_feature_values.duplicated(['machine_id', 'prediction_at']).any()

row_count_before_telemetry_features = len(gold_prediction_grid)
gold_prediction_grid = gold_prediction_grid.merge(
    telemetry_feature_values,
    on=['machine_id', 'prediction_at'],
    how='left',
    validate='one_to_one',
)

assert len(gold_prediction_grid) == row_count_before_telemetry_features
assert not gold_prediction_grid.duplicated(['machine_id', 'prediction_at']).any()
for source_column, feature_prefix in TELEMETRY_FEATURE_SOURCES.items():
    for hours in ROLLING_WINDOWS_HOURS:
        available_column = f'{feature_prefix}_available_count_{hours}h'
        missing_column = f'{feature_prefix}_missing_rate_{hours}h'
        assert gold_prediction_grid[available_column].between(0, hours).all()
        assert gold_prediction_grid[missing_column].between(0, 1).all()

feature_coverage_summary = []
for source_column, feature_prefix in TELEMETRY_FEATURE_SOURCES.items():
    for hours in ROLLING_WINDOWS_HOURS:
        feature_coverage_summary.append([
            feature_prefix,
            hours,
            gold_prediction_grid[f'{feature_prefix}_available_count_{hours}h'].mean(),
            gold_prediction_grid[f'{feature_prefix}_missing_rate_{hours}h'].mean(),
        ])

print(f'Features de télémétrie ajoutées : {len(telemetry_feature_columns)}')
print(tabulate(
    feature_coverage_summary,
    headers=['capteur', 'fenêtre (h)', 'mesures disponibles moyennes', 'taux de manque moyen'],
    tablefmt='github',
    floatfmt=('', 'd', '.4f', '.4%'),
))
print('Validation : 100 features calculées par machine, avec fenêtres exclusivement passées.')

Features de télémétrie ajoutées : 100
| capteur     |   fenêtre (h) |   mesures disponibles moyennes |   taux de manque moyen |
|-------------|---------------|--------------------------------|------------------------|
| temperature |             3 |                         2.9799 |                0.6698% |
| temperature |             6 |                         5.9588 |                0.6866% |
| temperature |            12 |                        11.9136 |                0.7202% |
| temperature |            24 |                        23.8110 |                0.7874% |
| pressure    |             3 |                         2.9777 |                0.7437% |
| pressure    |             6 |                         5.9544 |                0.7605% |
| pressure    |            12 |                        11.9047 |                0.7941% |
| pressure    |            24 |                        23.7933 |                0.8613% |
| voltage     |             3 |                         2.9997

### 5.2 Vérifier les lags avec une référence explicite

Un **lag disponible** à l'horizon `h` est la dernière valeur non nulle dont la date est inférieure ou égale à `prediction_at - h`. Son horodatage source et son âge réel sont contrôlés : une valeur plus ancienne que l'horizon reste utilisable, mais son nom ne doit pas masquer son ancienneté.

`delta` est une variation signée : `current_value - lag_value`. La variation relative est `delta / abs(lag_value)` lorsque le lag est présent et non nul. Nous calculons ces éléments par boucle pour `temperature_c` sur les trois snapshots de référence de `MACH-01`, avant toute vectorisation.

In [15]:
LAG_HORIZONS_HOURS = (1, 6, 24)

lag_reference_query = text(
    """
    SELECT measured_at, temperature_c
    FROM silver.telemetry
    WHERE machine_code = :machine_id
      AND measured_at <= :load_end
    ORDER BY measured_at
    """
)
engine = create_database_engine()
with engine.connect() as connection:
    lag_reference_telemetry = pd.read_sql(
        lag_reference_query,
        connection,
        params={
            'machine_id': REFERENCE_MACHINE_ID,
            'load_end': reference_prediction_times.max(),
        },
    )
engine.dispose()

def last_non_null_at_or_before(telemetry, timestamp):
    candidates = telemetry.loc[
        telemetry['measured_at'].le(timestamp)
        & telemetry['temperature_c'].notna()
    ]
    if candidates.empty:
        return None, pd.NaT
    last_row = candidates.iloc[-1]
    return last_row['temperature_c'], last_row['measured_at']

lag_reference_rows = []
for prediction_at in reference_prediction_times:
    current_value, current_source_at = last_non_null_at_or_before(
        lag_reference_telemetry, prediction_at
    )
    for hours in LAG_HORIZONS_HOURS:
        lag_deadline = prediction_at - pd.Timedelta(hours=hours)
        lag_value, lag_source_at = last_non_null_at_or_before(
            lag_reference_telemetry, lag_deadline
        )
        delta = (
            current_value - lag_value
            if current_value is not None and lag_value is not None
            else None
        )
        relative_change = (
            delta / abs(lag_value)
            if delta is not None and lag_value != 0
            else None
        )
        lag_reference_rows.append({
            'prediction_at': prediction_at,
            'lag_horizon_hours': hours,
            'temperature_current': current_value,
            'current_source_at': current_source_at,
            'current_age_hours': (prediction_at - current_source_at).total_seconds() / 3600
            if pd.notna(current_source_at) else None,
            'temperature_lag': lag_value,
            'lag_source_at': lag_source_at,
            'lag_actual_age_hours': (prediction_at - lag_source_at).total_seconds() / 3600
            if pd.notna(lag_source_at) else None,
            'temperature_delta': delta,
            'temperature_relative_change': relative_change,
        })

lag_reference_table = pd.DataFrame(lag_reference_rows)
available_lags = lag_reference_table['lag_source_at'].notna()
assert (
    lag_reference_table.loc[available_lags, 'lag_source_at']
    <= lag_reference_table.loc[available_lags, 'prediction_at']
    - pd.to_timedelta(lag_reference_table.loc[available_lags, 'lag_horizon_hours'], unit='h')
).all()
assert (lag_reference_table.loc[available_lags, 'lag_actual_age_hours']
        >= lag_reference_table.loc[available_lags, 'lag_horizon_hours']).all()
assert (lag_reference_table['current_age_hours'].dropna() >= 0).all()

print(tabulate(
    lag_reference_table,
    headers='keys',
    tablefmt='github',
    showindex=False,
    floatfmt='.4f',
))
print('Validation : lags disponibles, âges réels et variations utilisent uniquement le passé.')

| prediction_at             |   lag_horizon_hours |   temperature_current | current_source_at         |   current_age_hours |   temperature_lag | lag_source_at             |   lag_actual_age_hours |   temperature_delta |   temperature_relative_change |
|---------------------------|---------------------|-----------------------|---------------------------|---------------------|-------------------|---------------------------|------------------------|---------------------|-------------------------------|
| 2025-06-16 18:00:00+00:00 |                   1 |               69.5360 | 2025-06-16 18:00:00+00:00 |              0.0000 |           67.1460 | 2025-06-16 17:00:00+00:00 |                 1.0000 |              2.3900 |                        0.0356 |
| 2025-06-16 18:00:00+00:00 |                   6 |               69.5360 | 2025-06-16 18:00:00+00:00 |              0.0000 |           60.9630 | 2025-06-16 12:00:00+00:00 |                 6.0000 |              8.5730 |                     

### 5.2.1 Vectoriser les lags disponibles

`merge_asof` est une jointure temporelle : pour chaque échéance `prediction_at - h`, elle retrouve la dernière observation non nulle antérieure ou égale à cette échéance. L'argument `by='machine_id'` interdit tout emprunt de valeur à une autre machine.

Pour chaque capteur et les horizons 1 h, 6 h et 24 h, la grille reçoit la valeur de lag, son âge réel, le delta signé et la variation relative. Elle reçoit aussi le temps écoulé depuis sa dernière valeur non nulle. Les timestamps sources ne sont utilisés que pour les contrôles ; ils ne sont pas ajoutés au dataset.

In [16]:
def lookup_last_available(requests, valid_measurements, lookup_column):
    left = requests[['machine_id', 'prediction_at', lookup_column]].sort_values(
        [lookup_column, 'machine_id']
    )
    right = valid_measurements.sort_values(['source_at', 'machine_id'])
    return pd.merge_asof(
        left,
        right,
        left_on=lookup_column,
        right_on='source_at',
        by='machine_id',
        direction='backward',
        allow_exact_matches=True,
    )

lag_requests = gold_prediction_grid[['machine_id', 'prediction_at']].copy()
lag_feature_index = pd.MultiIndex.from_frame(lag_requests)
lag_feature_index = lag_feature_index.set_names(['machine_id', 'prediction_at'])
lag_feature_values_by_column = {}

for source_column, feature_prefix in TELEMETRY_FEATURE_SOURCES.items():
    valid_measurements = (
        telemetry_for_features[['machine_id', 'prediction_at', source_column]]
        .dropna(subset=[source_column])
        .rename(columns={
            'prediction_at': 'source_at',
            source_column: 'source_value',
        })
    )

    current_requests = lag_requests.copy()
    current_requests['current_deadline'] = current_requests['prediction_at']
    current_lookup = lookup_last_available(
        current_requests, valid_measurements, 'current_deadline'
    ).set_index(['machine_id', 'prediction_at']).reindex(lag_feature_index)
    current_source_at = current_lookup['source_at']
    current_value = current_lookup['source_value']
    existing_last = (
        gold_prediction_grid.set_index(['machine_id', 'prediction_at'])[f'{feature_prefix}_last']
        .reindex(lag_feature_index)
    )
    assert ((current_value.eq(existing_last)) | (current_value.isna() & existing_last.isna())).all()
    assert (current_source_at.dropna() <= lag_feature_index.get_level_values('prediction_at')[current_source_at.notna()]).all()
    lag_feature_values_by_column[f'{feature_prefix}_time_since_last_non_null_hours'] = (
        (pd.Series(lag_feature_index.get_level_values('prediction_at'), index=lag_feature_index)
         - current_source_at).dt.total_seconds() / 3600
    ).to_numpy()

    for hours in LAG_HORIZONS_HOURS:
        lag_requests_for_horizon = lag_requests.copy()
        lag_requests_for_horizon['lag_deadline'] = (
            lag_requests_for_horizon['prediction_at'] - pd.Timedelta(hours=hours)
        )
        lag_lookup = lookup_last_available(
            lag_requests_for_horizon, valid_measurements, 'lag_deadline'
        ).set_index(['machine_id', 'prediction_at']).reindex(lag_feature_index)
        lag_value = lag_lookup['source_value']
        lag_source_at = lag_lookup['source_at']
        lag_age_hours = (
            pd.Series(lag_feature_index.get_level_values('prediction_at'), index=lag_feature_index)
            - lag_source_at
        ).dt.total_seconds() / 3600
        delta = current_value - lag_value
        relative_change = (delta / lag_value.abs()).where(
            lag_value.notna() & lag_value.ne(0)
        )

        deadline = (
            pd.Series(lag_feature_index.get_level_values('prediction_at'), index=lag_feature_index)
            - pd.Timedelta(hours=hours)
        )
        assert (lag_source_at.dropna() <= deadline.loc[lag_source_at.notna()]).all()
        assert (lag_age_hours.dropna() >= hours).all()

        lag_feature_values_by_column[f'{feature_prefix}_lag_{hours}h'] = lag_value.to_numpy()
        lag_feature_values_by_column[f'{feature_prefix}_lag_actual_age_hours_{hours}h'] = (
            lag_age_hours.to_numpy()
        )
        lag_feature_values_by_column[f'{feature_prefix}_delta_{hours}h'] = delta.to_numpy()
        lag_feature_values_by_column[f'{feature_prefix}_relative_change_{hours}h'] = (
            relative_change.to_numpy()
        )

lag_feature_values = (
    pd.DataFrame(lag_feature_values_by_column, index=lag_feature_index)
    .reset_index()
)
lag_feature_columns = [
    column for column in lag_feature_values.columns
    if column not in {'machine_id', 'prediction_at'}
]
assert len(lag_feature_columns) == 52
assert not lag_feature_values.duplicated(['machine_id', 'prediction_at']).any()

row_count_before_lag_features = len(gold_prediction_grid)
gold_prediction_grid = gold_prediction_grid.merge(
    lag_feature_values,
    on=['machine_id', 'prediction_at'],
    how='left',
    validate='one_to_one',
)
assert len(gold_prediction_grid) == row_count_before_lag_features
assert not gold_prediction_grid.duplicated(['machine_id', 'prediction_at']).any()

vectorized_reference_rows = []
for expected in lag_reference_table.itertuples(index=False):
    snapshot = gold_prediction_grid.loc[
        gold_prediction_grid['machine_id'].eq(REFERENCE_MACHINE_ID)
        & gold_prediction_grid['prediction_at'].eq(expected.prediction_at)
    ].iloc[0]
    horizon = expected.lag_horizon_hours
    vectorized_reference_rows.append({
        'temperature_current': snapshot['temperature_last'],
        'current_age_hours': snapshot['temperature_time_since_last_non_null_hours'],
        'temperature_lag': snapshot[f'temperature_lag_{horizon}h'],
        'lag_actual_age_hours': snapshot[f'temperature_lag_actual_age_hours_{horizon}h'],
        'temperature_delta': snapshot[f'temperature_delta_{horizon}h'],
        'temperature_relative_change': snapshot[f'temperature_relative_change_{horizon}h'],
    })

reference_comparison_columns = [
    'temperature_current',
    'current_age_hours',
    'temperature_lag',
    'lag_actual_age_hours',
    'temperature_delta',
    'temperature_relative_change',
]
pd.testing.assert_frame_equal(
    pd.DataFrame(vectorized_reference_rows)[reference_comparison_columns],
    lag_reference_table[reference_comparison_columns],
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

lag_summary = [
    ['features de lag ajoutées', len(lag_feature_columns)],
    ['features Gold totales ajoutées dans 5.1 et 5.2',
     len(telemetry_feature_columns) + len(lag_feature_columns)],
]
print(tabulate(lag_summary, headers=['Contrôle', 'Valeur'], tablefmt='github'))
print('Validation : jointures temporelles par machine et référence MACH-01 identiques.')

| Contrôle                                       |   Valeur |
|------------------------------------------------|----------|
| features de lag ajoutées                       |       52 |
| features Gold totales ajoutées dans 5.1 et 5.2 |      152 |
Validation : jointures temporelles par machine et référence MACH-01 identiques.


## 5.3 — Tendances : exemple de référence avant vectorisation

Une **tendance** décrit l'évolution récente d'une mesure, au-delà de son niveau ponctuel et de ses lags. Pour chaque capteur, nous retenons une pente sur 6 h et 24 h, calculée par régression linéaire sur les valeurs non nulles de `(t - fenêtre ; t]`. La pente est exprimée dans l'unité du capteur par heure : elle est positive si la mesure augmente. Nous ajoutons aussi `mean_gap_6h_24h = moyenne_6h - moyenne_24h`, un signal simple d'accélération récente. Au moins deux observations valides sont nécessaires pour une pente ; sinon la valeur reste manquante.

L'exemple ci-dessous porte sur `MACH-01` et la température. Il vérifie que chaque calcul ne consomme que des mesures disponibles au plus tard à l'instant de prédiction `t`. Nous vectoriserons ensuite exactement cette convention pour les quatre capteurs.

In [17]:
import numpy as np

TREND_WINDOWS_HOURS = (6, 24)
trend_reference_rows = []

for prediction_at in reference_prediction_times:
    reference_row = {'prediction_at': prediction_at}
    window_means = {}

    for window_hours in TREND_WINDOWS_HOURS:
        window_measurements = lag_reference_telemetry.loc[
            lag_reference_telemetry['measured_at'].gt(
                prediction_at - pd.Timedelta(hours=window_hours)
            )
            & lag_reference_telemetry['measured_at'].le(prediction_at)
            & lag_reference_telemetry['temperature_c'].notna()
        ].copy()
        valid_count = len(window_measurements)
        window_means[window_hours] = window_measurements['temperature_c'].mean()

        if valid_count >= 2:
            elapsed_hours = (
                window_measurements['measured_at'] - prediction_at
            ).dt.total_seconds() / 3600
            slope = np.polyfit(elapsed_hours, window_measurements['temperature_c'], deg=1)[0]
        else:
            slope = np.nan

        reference_row[f'temperature_slope_{window_hours}h'] = slope
        reference_row[f'temperature_slope_valid_count_{window_hours}h'] = valid_count
        reference_row[f'temperature_slope_available_at_{window_hours}h'] = (
            window_measurements['measured_at'].max()
        )

    reference_row['temperature_mean_gap_6h_24h'] = (
        window_means[6] - window_means[24]
    )
    trend_reference_rows.append(reference_row)

trend_reference_table = pd.DataFrame(trend_reference_rows)
available_at_columns = [
    column for column in trend_reference_table.columns
    if column.startswith('temperature_slope_available_at_')
]
assert (trend_reference_table[available_at_columns].le(
    trend_reference_table['prediction_at'], axis=0
).all(axis=None))
assert (
    trend_reference_table[
        ['temperature_slope_valid_count_6h', 'temperature_slope_valid_count_24h']
    ].ge(2).all(axis=None)
)

display_columns = [
    'prediction_at',
    'temperature_slope_6h',
    'temperature_slope_valid_count_6h',
    'temperature_slope_24h',
    'temperature_slope_valid_count_24h',
    'temperature_mean_gap_6h_24h',
]
print(tabulate(
    trend_reference_table[display_columns],
    headers='keys',
    tablefmt='github',
    showindex=False,
    floatfmt='.4f',
))
print('Validation : chaque pente et chaque moyenne utilisent exclusivement des mesures datées au plus tard à t.')

| prediction_at             |   temperature_slope_6h |   temperature_slope_valid_count_6h |   temperature_slope_24h |   temperature_slope_valid_count_24h |   temperature_mean_gap_6h_24h |
|---------------------------|------------------------|------------------------------------|-------------------------|-------------------------------------|-------------------------------|
| 2025-06-16 18:00:00+00:00 |                 1.5409 |                                  6 |                  0.3153 |                                  24 |                        2.8117 |
| 2025-06-16 23:00:00+00:00 |                 2.2166 |                                  6 |                  0.7518 |                                  24 |                       10.3334 |
| 2025-06-17 00:00:00+00:00 |                 1.4955 |                                  6 |                  0.9437 |                                  24 |                       11.6663 |
Validation : chaque pente et chaque moyenne utilisent exclus

## 5.3 — Vectorisation des tendances

La pente d'une régression linéaire peut être calculée à partir de sommes : nombre de mesures valides, somme des temps, somme des valeurs, somme des temps au carré et somme `temps × valeur`. Des sommes glissantes par machine reproduisent donc la référence sans boucle ligne par ligne. Les temps sont convertis en heures uniquement pour obtenir une pente exprimée par heure ; déplacer tous les temps d'une même fenêtre ne modifie pas cette pente.

Nous calculons les deux pentes pour `temperature`, `pressure`, `voltage` et `rotation`, puis l'écart entre les moyennes 6 h et 24 h déjà produites à la section 5.1. Cela ajoute 12 features. Une pente reste `NaN` lorsqu'il y a moins de deux mesures valides ; les `available_count` existants permettent d'interpréter ce cas.

In [18]:
TREND_WINDOWS_HOURS = (6, 24)
trend_feature_values_by_column = {}
machine_first_measurement_at = telemetry_for_features.groupby('machine_id')['prediction_at'].transform('min')
time_in_hours = (
    telemetry_for_features['prediction_at'] - machine_first_measurement_at
).dt.total_seconds() / 3600
window_means_by_snapshot = gold_prediction_grid.set_index(
    ['machine_id', 'prediction_at']
)[[
    f'{feature_prefix}_mean_{hours}h'
    for feature_prefix in TELEMETRY_FEATURE_SOURCES.values()
    for hours in (6, 24)
]].reindex(feature_index)

for source_column, feature_prefix in TELEMETRY_FEATURE_SOURCES.items():
    sensor_value = telemetry_for_features[source_column]
    valid_time = time_in_hours.where(sensor_value.notna())
    trend_components = pd.DataFrame({
        'machine_id': telemetry_for_features['machine_id'],
        'prediction_at': telemetry_for_features['prediction_at'],
        'valid_count': sensor_value.notna().astype('int16'),
        'time_hours': valid_time,
        'value': sensor_value,
        'time_squared': valid_time ** 2,
        'time_value': valid_time * sensor_value,
    }).set_index('prediction_at')

    for window_hours in TREND_WINDOWS_HOURS:
        rolling_sums = (
            trend_components.groupby('machine_id')[
                ['valid_count', 'time_hours', 'value', 'time_squared', 'time_value']
            ]
            .rolling(window=f'{window_hours}h', closed='right')
            .sum()
        )
        rolling_sums.index = rolling_sums.index.set_names(['machine_id', 'prediction_at'])
        rolling_sums = rolling_sums.reindex(feature_index)

        valid_count = rolling_sums['valid_count']
        numerator = (
            valid_count * rolling_sums['time_value']
            - rolling_sums['time_hours'] * rolling_sums['value']
        )
        denominator = (
            valid_count * rolling_sums['time_squared']
            - rolling_sums['time_hours'] ** 2
        )
        trend_feature_values_by_column[f'{feature_prefix}_slope_{window_hours}h'] = (
            (numerator / denominator).where(valid_count.ge(2) & denominator.ne(0))
        ).to_numpy()

    trend_feature_values_by_column[f'{feature_prefix}_mean_gap_6h_24h'] = (
        window_means_by_snapshot[f'{feature_prefix}_mean_6h']
        - window_means_by_snapshot[f'{feature_prefix}_mean_24h']
    ).to_numpy()

trend_feature_values = pd.DataFrame(trend_feature_values_by_column, index=feature_index).reset_index()
trend_feature_columns = [
    column for column in trend_feature_values.columns
    if column not in {'machine_id', 'prediction_at'}
]
assert len(trend_feature_columns) == 12
assert not trend_feature_values.duplicated(['machine_id', 'prediction_at']).any()

row_count_before_trend_features = len(gold_prediction_grid)
gold_prediction_grid = gold_prediction_grid.merge(
    trend_feature_values,
    on=['machine_id', 'prediction_at'],
    how='left',
    validate='one_to_one',
)
assert len(gold_prediction_grid) == row_count_before_trend_features
assert not gold_prediction_grid.duplicated(['machine_id', 'prediction_at']).any()

vectorized_trend_reference = gold_prediction_grid.loc[
    gold_prediction_grid['machine_id'].eq(REFERENCE_MACHINE_ID)
    & gold_prediction_grid['prediction_at'].isin(reference_prediction_times),
    ['prediction_at', 'temperature_slope_6h', 'temperature_slope_24h', 'temperature_mean_gap_6h_24h'],
].sort_values('prediction_at').reset_index(drop=True)
explicit_trend_reference = trend_reference_table.loc[
    :, ['prediction_at', 'temperature_slope_6h', 'temperature_slope_24h', 'temperature_mean_gap_6h_24h']
].sort_values('prediction_at').reset_index(drop=True)
pd.testing.assert_frame_equal(
    vectorized_trend_reference,
    explicit_trend_reference,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

trend_summary = [
    ['features de tendance ajoutées', len(trend_feature_columns)],
    ['features Gold totales ajoutées dans 5.1 à 5.3',
     len(telemetry_feature_columns) + len(lag_feature_columns) + len(trend_feature_columns)],
]
print(tabulate(trend_summary, headers=['Contrôle', 'Valeur'], tablefmt='github'))
print('Validation : la vectorisation reproduit la référence MACH-01 à la tolérance de 1e-12.')

| Contrôle                                      |   Valeur |
|-----------------------------------------------|----------|
| features de tendance ajoutées                 |       12 |
| features Gold totales ajoutées dans 5.1 à 5.3 |      164 |
Validation : la vectorisation reproduit la référence MACH-01 à la tolérance de 1e-12.
